***AYUDANTIA 9***

Importar librerias

In [5]:
import numpy as np

**Forma Matricial**

Funciones

In [6]:
def prob_equilibrio(matriz_tasas):
    m_final = []

    # creamos y rellenamos matriz final
    # (matriz cuadrada llena de ceros que se llenará con las ecuaciones de equilibrio para cada estado)

    for i in range(len(matriz_tasas)):
        row = []
        for j in range(len(matriz_tasas)):
            row.append(0)
        m_final.append(row)

    t = 0
    #creamos matriz que representa ecuaciones de equilibrio
    for fila in matriz_tasas:
        #obtenemos tasas de salida
        sale = sum(fila)
        #ponemos la tasa de salida en la fila correspondiente al nodo
        m_final[t][t] = sale

        for h in range(len(matriz_tasas)):
            #si estamos en un nodo distinto y llega algo lo ponemos negativo (tasas de llegada)
            if h!= t and matriz_tasas[h][t] != 0:
                m_final[t][h] = -matriz_tasas[h][t]
        t+=1 

    #Agregar ecuacuion de normalizacion (sum P_i = 1)
    matriz_final = np.vstack((np.matrix(m_final),[[1]*len(m_final)]))
    vector_p = np.array([[0]]*(len(matriz_final)-1)+[[1]]).reshape(len(matriz_final))


    #Resolver sistema 
    P = np.linalg.lstsq(matriz_final,vector_p,rcond=-1)[0]

    return P


def imprimir(P):
    k = 0
    for fila in P:
        print(f"P_{k} = {fila}")
        k += 1

def esperanza(P):
    k = 0
    esp = 0
    for fila in P:
        esp += k* fila
        k+= 1
    return esp

Pregunta 1 - c)

In [7]:
# Definir parametros 
K = 5                   
gam = 2
thet = 1

# Crear matriz de tasas 
# (Cada entrada [i][j] representa la tasa a la que el sistema pasa del estado i al estado j)

matriz_tasas1 = [
    [0,(K+1)*gam,0,0,0,0,0],
    [thet,0,K*gam,0,0,0,0],
    [0,2*thet,0,(K-1)*gam,0,0,0],
    [0,0,3*thet,0,(K-2)*gam,0,0],
    [0,0,0,4*thet,0,(K-3)*gam,0],
    [0,0,0,0,5*thet,0,(K-4)*gam],
    [0,0,0,0,0,6*thet,0]
    ]


prob_parte_1 = prob_equilibrio(matriz_tasas1)
imprimir(prob_parte_1)
esperanza = esperanza(prob_parte_1)
print(f"Esperanza: {esperanza}")


P_0 = 0.0013717421124828497
P_1 = 0.016460905349794185
P_2 = 0.08230452674897101
P_3 = 0.21947873799725628
P_4 = 0.32921810699588455
P_5 = 0.2633744855967078
P_6 = 0.0877914951989025
Esperanza: 3.9999999999999973


Pregunta 1 - d)

In [8]:

Lambda = 1
mu = 2*Lambda


matriz_tasas2 = [
    [0,10*mu,0,10*mu,0,0,0,0,0],
    [Lambda,0,5*mu,0,10*mu,0,0,0,0],
    [0,2*Lambda,0,0,0,10*mu,0,0,0],
    [Lambda,0,0,0,10*mu,0,5*mu,0,0],
    [0,Lambda,0,Lambda,0,mu,0,mu,0],
    [0,0,Lambda,0,2*Lambda,0,0,0,mu],
    [0,0,0,2*Lambda,0,0,0,10*mu,0],
    [0,0,0,0,2*Lambda,0,Lambda,0,mu],
    [0,0,0,0,0,2*Lambda,0,2*Lambda,0]
    ]

prob_parte_2 = prob_equilibrio(matriz_tasas2)
imprimir(prob_parte_2)

P_0 = 0.000413736036408771
P_1 = 0.00827472072817541
P_2 = 0.014894497310715814
P_3 = 0.008274720728175448
P_4 = 0.21845262722383127
P_5 = 0.24493173355399273
P_6 = 0.014894497310715763
P_7 = 0.24493173355399256
P_8 = 0.2449317335539927


Pregunta 2 - a)

In [9]:
import numpy as np

def prob_equilibrio(matriz_tasas):
    """
    Calcula la distribución en régimen permanente de una CTMC
    dada su matriz de tasas (Q), resolviendo las ecuaciones de equilibrio
    y la condición de normalización.
    """
    n = len(matriz_tasas)
    # Construir matriz de coeficientes para el sistema lineal
    A = np.zeros((n+1, n))
    b = np.zeros(n+1)

    # Ecuaciones de equilibrio: P·Q = 0  <=>  para cada estado i: sum_j P_j*q_{j,i} = 0
    # Implementamos columnas de Q como tasas de llegada y filas como salidas.
    for i in range(n):
        # suma de tasas de salida desde el estado i
        salida = sum(matriz_tasas[i])
        A[i, i] = salida
        for j in range(n):
            if j != i and matriz_tasas[j][i] != 0:
                A[i, j] = -matriz_tasas[j][i]

    # Ecuación de normalización: sum_i P_i = 1
    A[n, :] = 1
    b[n] = 1

    # Resolver por mínimos cuadrados (o exactamente si la matriz es cuadrada)
    P, *_ = np.linalg.lstsq(A, b, rcond=None)
    return P

def imprimir_probabilidades(P, estados):
    """
    Imprime P_i para cada estado i, dado el vector P y la lista de estados.
    """
    for idx, estado in enumerate(estados):
        i, j = estado
        print(f"P({i},{j}) = {P[idx]:.6f}")

# Parámetros
lam = 1.0           # λ
mu  = 2 * lam       # μ = 2λ

# Enumeración de estados: (motores funcionando ala izquierda, ala derecha)
# Orden lexicográfico: (0,0), (0,1), ..., (2,2)
estados = [(i, j) for i in range(3) for j in range(3)]

# Construcción de la matriz de tasas Q (9×9)
Q = []
for (i, j) in estados:
    fila = []
    for (i2, j2) in estados:
        rate = 0.0

        # FALLA de un motor funcionando en ala izquierda
        if i > 0 and (i2, j2) == (i-1, j):
            rate = i * lam

        # FALLA de un motor funcionando en ala derecha
        elif j > 0 and (i2, j2) == (i, j-1):
            rate = j * lam

        # REPARACIÓN de un motor roto en ala izquierda
        elif i < 2 and (i2, j2) == (i+1, j):
            # ¿Se activa la reparación acelerada?
            mu_eff = 5 * mu if (i == 0 or j == 0) else mu
            motores_rotos_izq = 2 - i
            rate = motores_rotos_izq * mu_eff

        # REPARACIÓN de un motor roto en ala derecha
        elif j < 2 and (i2, j2) == (i, j+1):
            mu_eff = 5 * mu if (i == 0 or j == 0) else mu
            motores_rotos_der = 2 - j
            rate = motores_rotos_der * mu_eff

        fila.append(rate)
    Q.append(fila)

# Cálculo de las probabilidades de equilibrio
P = prob_equilibrio(Q)

# Impresión de todas las probabilidades
imprimir_probabilidades(P, estados)

# Fracción de tiempo con todos los motores funcionando: estado (2,2)
idx_full = estados.index((2, 2))
print(f"\nFracción de tiempo con todos los motores funcionando (P(2,2)): {P[idx_full]:.6f}")

P(0,0) = 0.000414
P(0,1) = 0.008275
P(0,2) = 0.014894
P(1,0) = 0.008275
P(1,1) = 0.218453
P(1,2) = 0.244932
P(2,0) = 0.014894
P(2,1) = 0.244932
P(2,2) = 0.244932

Fracción de tiempo con todos los motores funcionando (P(2,2)): 0.244932
